In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from torch.utils.data import Dataset
import transformers
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, TrainerCallback,
)
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

from meft import MeftConfig, MeftTrainer
from transformers import LlamaForCausalLM, LlamaTokenizer,ViTImageProcessor, ViTForImageClassification,EvalPrediction,TrainingArguments
from datasets import load_dataset
from transformers import BitsAndBytesConfig
from trl import SFTTrainer,SFTConfig

import logging
import socket
from datetime import datetime, timedelta

from torch.autograd.profiler import record_function
from utils.prompter import Prompter
import json
import numpy as np
import matplotlib.pyplot as plt
from torch.profiler._memory_profiler import _CATEGORY_TO_COLORS, _CATEGORY_TO_INDEX
import argparse
import evaluate
from pathlib import Path

/root/miniconda3/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
import torch
import numpy as np
from numpy.linalg import svd

model_name = "meta-llama/Llama-2-7b-hf"
tokenizer = LlamaTokenizer.from_pretrained(model_name)

tokenizer.pad_token_id = 0
tokenizer.padding_side = "left"

model = LlamaForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model.config.output_hidden_states = True  # 让 forward 返回隐藏状态（不影响 generate）

# config = LoraConfig(
#     r=64,
#     lora_alpha=16,
#     target_modules=["q_proj", "v_proj"],
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
# )
# model = get_peft_model(model, config)





Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
cutoff_len=256
prompter = Prompter("alpaca")
train_on_inputs=True
add_eos_token=True
data_path="yahma/alpaca-cleaned"
val_set_size=2000

In [4]:
def tokenize(prompt, add_eos_token=True):
    # there's probably a way to do this with the tokenizer settings
    # but again, gotta move fast
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=cutoff_len,
        padding=False,
        return_tensors=None,
    )
    if (
        result["input_ids"][-1] != tokenizer.eos_token_id
        and len(result["input_ids"]) < cutoff_len
        and add_eos_token
    ):
        result["input_ids"].append(tokenizer.eos_token_id)
        result["attention_mask"].append(1)

    result["labels"] = result["input_ids"].copy()

    return result

def generate_and_tokenize_prompt(data_point):
    full_prompt = prompter.generate_prompt(
        data_point["instruction"],
        data_point["input"],
        data_point["output"],
    )
    tokenized_full_prompt = tokenize(full_prompt)
    if not train_on_inputs:
        user_prompt = prompter.generate_prompt(
            data_point["instruction"], data_point["input"]
        )
        tokenized_user_prompt = tokenize(
            user_prompt, add_eos_token=add_eos_token
        )
        user_prompt_len = len(tokenized_user_prompt["input_ids"])

        if add_eos_token:
            user_prompt_len -= 1

        tokenized_full_prompt["labels"] = [
            -100
        ] * user_prompt_len + tokenized_full_prompt["labels"][
            user_prompt_len:
        ]  # could be sped up, probably
    return tokenized_full_prompt

if data_path.endswith(".json") or data_path.endswith(".jsonl"):
    data = load_dataset("json", data_files=data_path)
else:
    data = load_dataset(data_path)
    
# 载入数据
if data_path.endswith(".json") or data_path.endswith(".jsonl"):
    ds = load_dataset("json", data_files=data_path)      # DatasetDict
else:
    ds = load_dataset(data_path)                         # DatasetDict

orig_cols = ds["train"].column_names  # e.g. ['instruction','input','output',...]

# 先划分，再 map（或在无验证集时直接 map）
if val_set_size > 0:
    train_val = ds["train"].train_test_split(test_size=val_set_size, shuffle=True, seed=42)
    train_data = train_val["train"].shuffle(seed=42).map(
        generate_and_tokenize_prompt,
        remove_columns=orig_cols,     # 只保留 map 返回的键
    )
    val_data = train_val["test"].shuffle(seed=42).map(
        generate_and_tokenize_prompt,
        remove_columns=orig_cols,
    )
else:
    train_data = ds["train"].shuffle(seed=42).map(
        generate_and_tokenize_prompt,
        remove_columns=orig_cols,
    )
    val_data = None

print("slicing train_data to 320 for testing")
train_data = train_data.select(range(320))
print("after slicing, train_data:", train_data)


slicing train_data to 320 for testing
after slicing, train_data: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 320
})


In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq
from numpy.linalg import svd

# ❌ 删掉：accelerate 下不要手动搬模型
# device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
# model.to(device)

model.eval()

# pad_token 兜底
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
    return_tensors="pt",
    label_pad_token_id=-100,
)

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=False,
    collate_fn=collator,
    drop_last=False,
)

# 关键：推断“前向所需设备”（嵌入层所在设备，accelerate 场景下通常是 cuda:0）
def infer_forward_device(m):
    try:
        return m.get_input_embeddings().weight.device
    except Exception:
        for p in m.parameters():
            return p.device
    return torch.device("cpu")

fwd_device = infer_forward_device(model)

batch = next(iter(train_loader))

# 把需要的输入搬到“嵌入层的设备”即可；不要把 labels、别的键乱搬
inputs = {k: v.to(fwd_device) for k, v in batch.items() if k in ("input_ids", "attention_mask")}

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

hidden_states = outputs.hidden_states
target_layer = 10
assert target_layer < len(hidden_states), f"模型只有 {len(hidden_states)} 层"

hs = hidden_states[target_layer]  # [B, T, H]
B, T, H = hs.shape
hs2d = hs.reshape(B*T, H)

hs_np = hs2d.to(torch.float32).cpu().numpy()
U, s, Vt = svd(hs_np, full_matrices=False)
print(hs2d.shape, U.shape, s.shape, Vt.shape)


torch.Size([8192, 4096]) (8192, 4096) (4096,) (4096, 4096)


In [6]:
print(s)

[1.7251408e+04 4.6203168e+02 3.3087759e+02 ... 6.7956924e-01 6.7203474e-01
 6.6078430e-01]
